Use beta tool and tool_run instead of writing our own loop

In [2]:


from anthropic import beta_tool
import json

@beta_tool
def create_calendar_event(
    title: str,
    start: str,
    end: str,
    attendees: list[str] | None = None,
    recurrence: dict | None = None,
) -> str:
    """Create a calendar event with attendees and optional recurrence.

    Args:
        title: Event title.
        start: Start time in ISO 8601 format.
        end: End time in ISO 8601 format.
        attendees: Email addresses to invite.
        recurrence: Dict with 'frequency' (daily, weekly, monthly) and 'count'.
    """
    if attendees and len(attendees) > 10:
        raise ValueError("Too many attendees (max 10)")
    return json.dumps({"event_id": "evt_123", "status": "created", "title": title})

@beta_tool
def list_calendar_events(date: str) -> str:
    """List all calendar events on a given date.

    Args:
        date: Date in YYYY-MM-DD format.
    """
    return json.dumps({"events": [{"title": "Existing meeting", "start": "14:00", "end": "15:00"}]})


In [3]:
from util.claude_client import  get_claude_client_and_default_model

client , llm= get_claude_client_and_default_model('claude')
final_message = client.beta.messages.tool_runner(
    model=llm,
    max_tokens=1024,
    tools=[create_calendar_event, list_calendar_events],
    messages=[
        {
            "role": "user",
            "content": "Check what I have on Monday, March 30, 2026, then schedule a one-hour planning session that day that avoids any conflicts.",
        }
    ],
).until_done()

for block in final_message.content:
    if block.type == "text":
        print(block.text)

llm profile name : claude , model : claude-haiku-4-5-20251001
Perfect! I've scheduled your one-hour planning session for **Monday, March 30, 2026 from 10:00 AM to 11:00 AM**. This avoids the existing meeting you have from 2:00 PM to 3:00 PM.


show execution  details

In [4]:
runner = client.beta.messages.tool_runner(
    model="claude-opus-5-5",
    max_tokens=1024,
    max_iterations=10,  # optional safety bound
    tools=[create_calendar_event, list_calendar_events],
    messages=[{"role": "user", "content": "Check what I have on Monday, March 30, 2026, then schedule a one-hour planning session that day that avoids any conflicts."}],
)

for i, message in enumerate(runner):
    print(f"--- iteration {i} | stop_reason={message.stop_reason} ---")

    # Log the assistant's content (text + tool_use blocks)
    for block in message.content:
        if block.type == "text":
            print("assistant text:", block.text)
        elif block.type == "tool_use":
            print(f"tool call: {block.name}({json.dumps(block.input)})")

    # Log the tool results the runner will send back next
    tool_response = runner.generate_tool_call_response()
    if tool_response is not None:
        for block in tool_response["content"]:
            print("tool result:", block.get("content"), "| is_error:", block.get("is_error", False))

--- iteration 0 | stop_reason=tool_use ---
tool call: list_calendar_events({"date": "2026-03-30"})
tool result: {"events": [{"title": "Existing meeting", "start": "14:00", "end": "15:00"}]} | is_error: False
--- iteration 1 | stop_reason=tool_use ---
tool call: create_calendar_event({"title": "Planning Session", "start": "2026-03-30T10:00:00", "end": "2026-03-30T11:00:00"})
tool result: {"event_id": "evt_123", "status": "created", "title": "Planning Session"} | is_error: False
--- iteration 2 | stop_reason=end_turn ---
assistant text: I've added a one-hour **Planning Session** to your calendar on Monday, March 30, 2026, from **10:00 to 11:00 AM**.

You only had one other thing that day, a meeting from 2:00 to 3:00 PM, so the two don't overlap. I didn't invite anyone to the session. I can add people or move it to another open time if you'd like.
